In [2]:

import asyncio
import warnings
import pandas as pd
from pathlib import Path

# Парсеры
from antimony import antimony_parser
from westmetall import westmetall_async
from lme import lme_selenium_async
from lbma import lbma_prescious_async

from cbr import cb_currency, cb_metalls
from nbk import nbk_tenge_async
from shmet import shmet_optimized_async
from kitco import kitco_parser_async

# Сервисные функции из service_layer
from service_layer import (
    read_db,
    save_db,
    check_df,
    check_and_save_pair,
    show_db,
    excel_to_csv_db
)

warnings.filterwarnings("ignore")


# ================================================================
# Пути к базам (реальная структура проекта)
# ================================================================
LME_PATH = Path("lme/data/LME_db_new.xlsx")
WESTMETALL_PATH = Path("westmetall/data/LME_westmetall_db.xlsx")

KITCO_PATH = Path("kitco/data/kitko_db.xlsx")
LBMA_PATH = Path("lbma/data/lbma_kitco_subs.xlsx")

ANTIMONY_PATH = Path("antimony/data/antimony.xlsx")

CB_CURRENCY_PATH = Path("cbr/data/cb_currency.xlsx")
CB_METALLS_PATH = Path("cbr/data/cb_metalls.xlsx")

NBK_PATH = Path("nbk/data/nbk_tenge.xlsx")
SHMET_PATH = Path("shmet/data/shmet_historical.xlsx")


# ================================================================
# Проверка целостности парных баз
# ================================================================
def db_check():
    """
    Проверка целостности парных баз с очисткой дубликатов
    и сохранением первого (более раннего) вхождения.
    """
    print("Проверка LME / Westmetall...")
    check_and_save_pair(
        LME_PATH,
        WESTMETALL_PATH,
        pair_name="LME / Westmetall",
        index=False,
    )

    print("Проверка Kitco / LBMA...")
    check_and_save_pair(
        KITCO_PATH,
        LBMA_PATH,
        pair_name="Kitco / LBMA",
        index=False,
    )
    
async def main():
    print("Parsing started...")

    tasks = {
        "lme": lme_selenium_async(),
        "antimony": antimony_parser(),
        "westmetall": westmetall_async(),
        "lbma": lbma_prescious_async(),
        "kitco": kitco_parser_async(),
        "cb_currency": cb_currency(),
        "cb_metalls": cb_metalls(),
        "nbk": nbk_tenge_async(),
        "shmet": shmet_optimized_async(),
    }

    # return_exceptions=True — чтобы падение одного парсера
    # не останавливало остальные
    results = await asyncio.gather(
        *tasks.values(),
        return_exceptions=True,
    )

    for name, result in zip(tasks.keys(), results):
        if isinstance(result, Exception):
            print(f"❌ Ошибка в {name}: {result}")

    print("All tasks are done!")
    print("+" * 64)
    print("Checking DB...")

    db_check()

    print("DB check completed!")
    print("+" * 64)
    print("Visual control")
    print("+" * 64)
    
    print("Converting Excel to CSV...")
    excel_to_csv_db()
    print("CSV conversion completed!")

    # Базовые металлы
    show_db("lme_selenium_db", LME_PATH, sheet_name=0)
    show_db("westmetall_db", WESTMETALL_PATH, sheet_name=0)

    # Драгоценные металлы
    show_db("kitco_db", KITCO_PATH, sheet_name=0)
    show_db("lbma_precious_db", LBMA_PATH, sheet_name=0)

    # Антимоний
    show_db("antimony_db", ANTIMONY_PATH, sheet_name=0)

    # ЦБ РФ: валюты (каждая на своем листе)
    for currency in [
        "USD",
        "EUR",
        "British_Pound",
        "China_Yuan",
        "Japanese_Yen",
        "Swiss_Franc",
    ]:
        show_db(
            f"cb_currency ({currency})",
            CB_CURRENCY_PATH,
            sheet_name=currency,
        )

    # ЦБ РФ: металлы
    show_db("cb_metalls_db", CB_METALLS_PATH, sheet_name=0)

    # Казахстан и SHMET
    show_db("nbk_tenge_db", NBK_PATH, sheet_name=0)
    show_db("shmet_historical_db", SHMET_PATH, sheet_name=0, show_head=True)


# Для Jupyter используем await, а не asyncio.run()
await main()

Parsing started...
🚀 LME parsing started...
antimony parsing is DONE
NBK_tenge parsing is DONE! (1753 строк)
✅ LME_main is done!!!
CB_metalls parsing is DONE!
Произошла ошибка KITCO: Не найдены блоки <div class='grid'> на странице Kitco
USD is done!
EUR is done!
Australian_Dollar is done!
China_Yuan is done!
British_Pound is done!
Kazakhstan_Tenge is done!
Japanese_Yen is done!
Swiss_Franc is done!
CB_currency parsing is DONE!
SHMET is done!!!
WESTMETALL is done!!!
[LBMA/Selenium] Открываем страницу LBMA...
[LBMA/Selenium] Обрабатываем Gold...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Gold: получено 10 строк
[LBMA/Selenium] Обрабатываем Silver...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 1
[LBMA/Selenium] Silver: получено 10 строк
[LBMA/Selenium] Обрабатываем Platinum...
[LBMA/Selenium] Сетевых ответов от prices.lbma.org.uk: 2
[LBMA/Selenium] Platinum: получено 10 строк
[LBMA/Selenium] Обрабатываем Palladium...
[LBMA/Selenium] Сетевых ответов

,date,lead,nickel,zink,tin,aluminium,copper
1230,2026-11-13,1853.0,16025,3745.0,48295,3150.5,13610.0
1231,2026-11-16,1856.0,15805,3848.0,46895,3332.0,13375.0
1232,2026-11-17,1840.0,15790,3776.0,46650,3060.5,13225.5
1233,2026-11-18,1831.0,15800,3732.0,42050,3095.0,12750.0
1234,2026-11-19,1822.0,15425,3714.5,42200,3105.0,12570.0


westmetall_db


,date,aluminium,copper,lead,nickel,zink,tin
1230,2026-11-13,3150.5,13610.0,1853.0,16025,3745.0,48295
1231,2026-11-16,3332.0,13375.0,1856.0,15805,3848.0,46895
1232,2026-11-17,3060.5,13225.5,1840.0,15790,3776.0,46650
1233,2026-11-18,3095.0,12750.0,1831.0,15800,3732.0,42050
1234,2026-11-19,3105.0,12570.0,1822.0,15425,3714.5,42200


kitco_db


,Date,Gold,Silver,Platinum,Palladium
14891,2026-11-13,3890.85,50.045,1507.65,1013.70
14892,2026-11-16,3878.25,47.730,1463.40,988.00
14893,2026-11-17,3858.10,47.700,1437.05,986.50
14894,2026-11-18,3759.35,45.720,1384.10,961.35
14895,2026-11-19,3709.50,45.020,1381.45,943.75


... (18 outputs left out)

nbk_tenge_db


,date,Числовое значение,ДОЛЛАР США
1748,2026-11-16,1,444.30
1749,2026-11-17,1,444.40
1750,2026-11-18,1,443.96
1751,2026-11-19,1,444.16
1752,2026-11-20,1,448.72


shmet_historical_db


,date,price,unit
0,2020-01-10,48605,Yuan/MT
1,2020-01-14,48990,Yuan/MT
2,2020-01-15,49060,Yuan/MT
3,2020-01-16,48950,Yuan/MT
4,2020-01-17,48930,Yuan/MT
